# 00 Data Exploration

## Purpose

This is the first notebook in the rebuilt workflow. I am using it to inspect the raw dataset before making any splits.

## Inputs

- `MyDrive/ProjectRoot/data/raw/raw_glycans_dataset_no_aldi.txt`

## Outputs

- quick printed dataset summary
- sequence length distribution plot
- summary CSV saved to Drive

## Notes to myself

This notebook should stay lightweight. I just want to confirm the raw file is correct, look at sequence lengths, and save a basic reference figure.

## Why the setup cell looks like this

I am separating code from heavy outputs right away so the workflow is easier to reproduce.

- the code and notebooks live in GitHub
- the raw data and generated outputs live in Drive
- Colab pulls the repo at the start, then I sync the notebook back to GitHub at the end

The repo stays clean and Drive holds the large files.

In [ ]:
# ==============================================================================
# 0. SET UP THE COLAB ENVIRONMENT
# ==============================================================================
import os
import sys

from google.colab import drive, userdata

# Mount Google Drive so the notebook can read the raw dataset and save results.
drive.mount('/content/drive')

# Define the GitHub repo used for the rebuilt workflow.
GITHUB_USER = os.environ.get('GLYCAN_GITHUB_OWNER', 'hb791-dev')
REPO_NAME = 'glycan-roberta'
GITHUB_EMAIL = os.environ.get('GLYCAN_GITHUB_EMAIL', f'{GITHUB_USER}@users.noreply.github.com')
GITHUB_NAME = os.environ.get('GLYCAN_GITHUB_NAME', GITHUB_USER)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

# Clone the repo if it is not in the current runtime. Otherwise pull the latest version.
if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    print('Repository already exists. Pulling latest changes...')

%cd {REPO_DIR}

!git config --global user.email "{GITHUB_EMAIL}"
!git config --global user.name "{GITHUB_NAME}"
!git config --global pull.rebase false
!git pull {REPO_URL} main --no-edit -q

# Add the repo to the Python path so src/ imports work across notebooks.
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

print('Colab environment ready.')
print(f'Repo directory: {REPO_DIR}')

## Path setup

I want the important paths defined once near the top. For this notebook, the main things I care about are:

- where the new Drive project root lives
- where the raw data file lives
- where the exploration outputs should go

I am also creating the results folder here instead of assuming it already exists.

In [ ]:
# ==============================================================================
# 1. DEFINE THE DRIVE PATHS FOR THIS NOTEBOOK
# ==============================================================================
PROJECT_ROOT = '/content/drive/MyDrive/ProjectRoot'
RAW_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'raw', 'raw_glycans_dataset_no_aldi.txt')
EXPLORATION_RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'exploration')

os.makedirs(EXPLORATION_RESULTS_DIR, exist_ok=True)

print('Project root:')
print(PROJECT_ROOT)
print('\nRaw data path:')
print(RAW_DATA_PATH)
print('\nExploration results directory:')
print(EXPLORATION_RESULTS_DIR)

if not os.path.exists(RAW_DATA_PATH):
    raise FileNotFoundError(f'Raw dataset not found: {RAW_DATA_PATH}')

## Basic dataset summary

At this stage I want to answer a few simple questions:

- how many sequences are in the file
- what do a few example sequences look like
- how long are the sequences in characters

I am using character length here because tokenizers do not exist yet. Later, token length will matter more.

In [ ]:
# ==============================================================================
# 2. LOAD THE RAW DATASET AND COMPUTE BASIC SUMMARY STATISTICS
# ==============================================================================
import numpy as np
import pandas as pd

with open(RAW_DATA_PATH, 'r', encoding='utf-8') as file:
    glycan_sequences = [line.strip() for line in file if line.strip()]

sequence_lengths = np.array([len(sequence) for sequence in glycan_sequences])

dataset_summary = pd.DataFrame(
    {
        'metric': [
            'num_sequences',
            'min_char_length',
            'mean_char_length',
            'median_char_length',
            'max_char_length',
            'p95_char_length',
            'p99_char_length',
        ],
        'value': [
            len(glycan_sequences),
            int(sequence_lengths.min()),
            float(sequence_lengths.mean()),
            float(np.median(sequence_lengths)),
            int(sequence_lengths.max()),
            float(np.percentile(sequence_lengths, 95)),
            float(np.percentile(sequence_lengths, 99)),
        ],
    }
)

print('Dataset summary')
display(dataset_summary)

preview_count = min(5, len(glycan_sequences))
preview_df = pd.DataFrame(
    {
        'example_index': list(range(preview_count)),
        'sequence': glycan_sequences[:preview_count],
        'char_length': [len(sequence) for sequence in glycan_sequences[:preview_count]],
    }
)

print('\nExample sequences')
display(preview_df)

## Sequence length plot

This plot is here to show whether the raw dataset has a long tail of very large glycans. That matters later when I choose padding and truncation rules.

What I am looking for:

- whether most glycans are fairly compact
- whether there are a few very long outliers
- whether the distribution looks stable enough that percentile-based length cutoffs will make sense later

In [ ]:
# ==============================================================================
# 3. PLOT THE RAW SEQUENCE LENGTH DISTRIBUTION
# ==============================================================================
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.hist(sequence_lengths, bins=40, color='steelblue', edgecolor='black', alpha=0.85)
plt.xlabel('Sequence length (characters)')
plt.ylabel('Count')
plt.title('Raw glycan sequence length distribution')
plt.grid(alpha=0.25)

plot_path = os.path.join(EXPLORATION_RESULTS_DIR, 'sequence_length_distribution.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()

print(f'Plot saved to: {plot_path}')

## Save lightweight outputs

I want this notebook to leave behind a small paper trail in Drive. Saving the summary now makes later comparisons easier.

In [ ]:
# ==============================================================================
# 4. SAVE THE SUMMARY TABLES
# ==============================================================================
summary_path = os.path.join(EXPLORATION_RESULTS_DIR, 'dataset_summary.csv')
preview_path = os.path.join(EXPLORATION_RESULTS_DIR, 'example_sequences.csv')

dataset_summary.to_csv(summary_path, index=False)
preview_df.to_csv(preview_path, index=False)

print(f'Dataset summary saved to: {summary_path}')
print(f'Example sequence preview saved to: {preview_path}')

## GitHub sync note

The notebook itself stays in GitHub even though the outputs live in Drive. This final cell just syncs the notebook back into the repo clone.

In [ ]:
# ==============================================================================
# SAVE THE NOTEBOOK BACK TO GITHUB
# ==============================================================================
import json

REPO_NOTEBOOK_PATH = os.path.join(REPO_DIR, 'notebooks/00_data_exploration.ipynb')
NOTEBOOK_FILENAME = os.path.basename(REPO_NOTEBOOK_PATH)
DRIVE_NOTEBOOK_CANDIDATES = [
    f'/content/drive/MyDrive/Colab Notebooks/{NOTEBOOK_FILENAME}',
    f'/content/drive/MyDrive/{NOTEBOOK_FILENAME}',
]

source_notebook_path = None
for candidate in DRIVE_NOTEBOOK_CANDIDATES:
    if os.path.exists(candidate):
        source_notebook_path = candidate
        break

if source_notebook_path is not None:
    !cp "{source_notebook_path}" "{REPO_NOTEBOOK_PATH}"

    # Strip widget metadata if Colab adds it so GitHub rendering stays cleaner.
    try:
        with open(REPO_NOTEBOOK_PATH, 'r', encoding='utf-8') as file:
            notebook_json = json.load(file)

        if 'widgets' in notebook_json.get('metadata', {}):
            del notebook_json['metadata']['widgets']

        with open(REPO_NOTEBOOK_PATH, 'w', encoding='utf-8') as file:
            json.dump(notebook_json, file, indent=1)
    except Exception as exc:
        print(f'Notebook metadata cleanup skipped: {exc}')

    %cd {REPO_DIR}
    !git add notebooks/00_data_exploration.ipynb
    !git commit -m "Update 00_data_exploration" || echo "No new changes to commit."
    !git pull {REPO_URL} main --no-edit -q
    !git push {REPO_URL} main -q

    print('Notebook synced to GitHub.')
    print(f'Source notebook path: {source_notebook_path}')
else:
    print('No Drive-backed notebook file was found for this session.')
    print('If you opened this notebook directly from GitHub, Colab is editing a browser copy, not a runtime file.')
    print('For GitHub-opened notebooks, use File -> Save a copy in GitHub.')
    print('If you want this cell to auto-sync the notebook, first save or copy the notebook into Drive and then rerun this cell.')
